# Hospital Readmissions Reduction Program Analysis

## 1. Data Loading and Validation

In [12]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/readmission_reduction.csv",
    dtype={"Facility ID": "string"}
)

In [13]:
df.shape

(18510, 12)

### Initial Raw Data Checks

In [14]:
df["Facility ID"].dtype

<StringDtype(na_value=<NA>)>

In [15]:
print("Facility ID dtype:", df["Facility ID"].dtype)
print("\nFirst 5 Facility IDs:")
print(df["Facility ID"].head())

print("\nFacility ID lengths:")
print(df["Facility ID"].str.len().value_counts().sort_index())

Facility ID dtype: string

First 5 Facility IDs:
0    010001
1    010001
2    010001
3    010001
4    010001
Name: Facility ID, dtype: string

Facility ID lengths:
Facility ID
6    18510
Name: count, dtype: Int64


In [16]:
print("Unique facilities:", df["Facility ID"].nunique())

print("\nMeasure counts:")
print(df["Measure Name"].value_counts())

print("\nRows with usable ERR:",
      df["Excess Readmission Ratio"].notna().sum())

Unique facilities: 3085

Measure counts:
Measure Name
READM-30-AMI-HRRP         3085
READM-30-CABG-HRRP        3085
READM-30-HF-HRRP          3085
READM-30-HIP-KNEE-HRRP    3085
READM-30-PN-HRRP          3085
READM-30-COPD-HRRP        3085
Name: count, dtype: int64

Rows with usable ERR: 11927


### Condition Mapping and Validation

In [17]:
condition_map = {
    "READM-30-AMI-HRRP": "Medical",
    "READM-30-HF-HRRP": "Medical",
    "READM-30-PN-HRRP": "Medical",
    "READM-30-COPD-HRRP": "Medical",
    "READM-30-CABG-HRRP": "Surgical",
    "READM-30-HIP-KNEE-HRRP": "Surgical"
}

expected_measures = set(condition_map.keys())
actual_measures = set(df["Measure Name"].dropna().unique())

assert actual_measures == expected_measures, (
    f"Measure mismatch. "
    f"Missing from mapping: {actual_measures - expected_measures}. "
    f"Unexpected in mapping: {expected_measures - actual_measures}."
)

print("Validation passed: all CMS measures are included in the condition mapping.")

Validation passed: all CMS measures are included in the condition mapping.


In [18]:
df["Condition Type"] = df["Measure Name"].map(condition_map)

print("Condition Type counts:")
print(df["Condition Type"].value_counts())

print("\nUnmapped records:", df["Condition Type"].isna().sum())

assert df["Condition Type"].notna().all(), (
    "Validation failed: one or more records were not assigned a Condition Type."
)

print("\nValidation passed: every record was assigned a Condition Type.")

Condition Type counts:
Condition Type
Medical     12340
Surgical     6170
Name: count, dtype: int64

Unmapped records: 0

Validation passed: every record was assigned a Condition Type.


## 2. Analysis Dataset

In [20]:
analysis_df = df.dropna(subset=["Excess Readmission Ratio"]).copy()

print("Usable ERR records:", len(analysis_df))
print("Unique facilities:", analysis_df["Facility ID"].nunique())

print("\nCondition Type counts:")
print(analysis_df["Condition Type"].value_counts())

print("\nUnmapped records:",
      analysis_df["Condition Type"].isna().sum())

Usable ERR records: 11927
Unique facilities: 2862

Condition Type counts:
Condition Type
Medical     9456
Surgical    2471
Name: count, dtype: int64

Unmapped records: 0


### Descriptive Statistics by Condition Type

In [21]:
summary = (
    analysis_df
    .groupby("Condition Type")["Excess Readmission Ratio"]
    .agg(["mean", "median", "std", "count"])
)

print(summary)

                    mean  median       std  count
Condition Type                                   
Medical         1.001469  0.9986  0.064060   9456
Surgical        1.002678  0.9946  0.124963   2471


### Welch's Two-Sample T-Test

In [23]:
from scipy.stats import ttest_ind

medical = analysis_df.loc[
    analysis_df["Condition Type"] == "Medical",
    "Excess Readmission Ratio"
]

surgical = analysis_df.loc[
    analysis_df["Condition Type"] == "Surgical",
    "Excess Readmission Ratio"
]

t_stat, p_value = ttest_ind(
    medical,
    surgical,
    equal_var=False
)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

T-statistic: -0.4650
P-value: 0.6419


### Records Above Expected Readmission Levels

In [24]:
analysis_df["Above Expected"] = (
    analysis_df["Excess Readmission Ratio"] > 1
)

print(analysis_df["Above Expected"].value_counts())

print("\nPercent above expected:")
print(
    analysis_df["Above Expected"].value_counts(normalize=True) * 100
)

Above Expected
False    6113
True     5814
Name: count, dtype: int64

Percent above expected:
Above Expected
False    51.253459
True     48.746541
Name: proportion, dtype: float64


In [25]:
facility_summary = (
    analysis_df
    .groupby(["Facility ID", "Facility Name"])
    .agg(
        usable_conditions=("Excess Readmission Ratio", "count"),
        above_expected_conditions=("Above Expected", "sum"),
        mean_err=("Excess Readmission Ratio", "mean")
    )
    .reset_index()
)

facility_summary.head()

,Facility ID,Facility Name,usable_conditions,above_expected_conditions,mean_err
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,6,1,0.971467
1,010005,MARSHALL MEDICAL CENTERS,4,0,0.890575
2,010006,NORTH ALABAMA MEDICAL CENTER,6,2,0.992550
3,010007,MIZELL MEMORIAL HOSPITAL,4,3,1.028350
4,010011,ST. VINCENT'S EAST,6,4,1.025167


In [26]:
facility_distribution = (
    facility_summary["above_expected_conditions"]
    .value_counts()
    .sort_index()
)

print(facility_distribution)

above_expected_conditions
0    487
1    681
2    639
3    561
4    334
5    124
6     36
Name: count, dtype: int64


In [27]:
facilities_3plus = (
    facility_summary["above_expected_conditions"] >= 3
).sum()

total_facilities = len(facility_summary)

percent_3plus = (
    facilities_3plus / total_facilities * 100
)

print("Facilities with 3+ conditions above expected:", facilities_3plus)
print("Total facilities:", total_facilities)
print(f"Percent with 3+ conditions above expected: {percent_3plus:.2f}%")

Facilities with 3+ conditions above expected: 1055
Total facilities: 2862
Percent with 3+ conditions above expected: 36.86%


### Concentration of Above-Expected ERR Records Across Facilities

In [28]:
facility_concentration = (
    facility_summary
    .sort_values("above_expected_conditions", ascending=False)
    .copy()
)

total_above_expected = facility_concentration["above_expected_conditions"].sum()

facility_concentration["cumulative_above_expected"] = (
    facility_concentration["above_expected_conditions"].cumsum()
)

facility_concentration["cumulative_share"] = (
    facility_concentration["cumulative_above_expected"]
    / total_above_expected
    * 100
)

facilities_for_80 = (
    facility_concentration["cumulative_share"] < 80
).sum() + 1

percent_facilities_for_80 = (
    facilities_for_80 / len(facility_concentration) * 100
)

print("Total above-expected condition records:", total_above_expected)
print("Facilities needed to account for 80%:", facilities_for_80)
print(
    f"Share of facilities needed to account for 80%: "
    f"{percent_facilities_for_80:.2f}%"
)

Total above-expected condition records: 5814
Facilities needed to account for 80%: 1454
Share of facilities needed to account for 80%: 50.80%


### Concentration of ERR Excess Above 1.0

In [29]:
analysis_df["ERR Excess"] = (
    analysis_df["Excess Readmission Ratio"] - 1
).clip(lower=0)

facility_excess = (
    analysis_df
    .groupby(["Facility ID", "Facility Name"])
    .agg(total_err_excess=("ERR Excess", "sum"))
    .reset_index()
    .sort_values("total_err_excess", ascending=False)
)

total_err_excess = facility_excess["total_err_excess"].sum()

facility_excess["cumulative_excess"] = (
    facility_excess["total_err_excess"].cumsum()
)

facility_excess["cumulative_share"] = (
    facility_excess["cumulative_excess"]
    / total_err_excess
    * 100
)

facilities_for_80_excess = (
    facility_excess["cumulative_share"] < 80
).sum() + 1

percent_facilities_for_80_excess = (
    facilities_for_80_excess / len(facility_excess) * 100
)

print("Total ERR excess above 1.0:", round(total_err_excess, 4))
print("Facilities needed to account for 80% of ERR excess:",
      facilities_for_80_excess)
print(
    f"Share of facilities needed to account for 80% of ERR excess: "
    f"{percent_facilities_for_80_excess:.2f}%"
)

Total ERR excess above 1.0: 357.8068
Facilities needed to account for 80% of ERR excess: 1057
Share of facilities needed to account for 80% of ERR excess: 36.93%
